In [32]:
import pandas as pd 
import re
import emoji 
from pathlib import Path 
data_dir = Path('../data')
resume_dataset=pd.read_csv(data_dir/'interim/combined_resume_final.csv')
# resume_dataset2=pd.read_csv(data_dir/'raw/UpdatedResumeDataSet.csv')


In [33]:
#Lecture et affichage Information dataframme
def read_data(data,column_name):
    print("\n __Aperçu des données :")
    print(data.head(3))
    # Afficher les informations sur le DataFrame
    print("\n __Informations sur les données :")
    print(data.info())
    print(" \n __Taille : \n",data.shape)
    print("\n __Information sur les Categories: \n",data[column_name].value_counts().reset_index())

## 1. DATASET = RESUME

#### Lecture et affichage des donnees

In [34]:
resume_dataset.head()

,Category,Resume
0,Database Administrator,"Ability: Installation and Building Server, Run..."
1,Database Administrator,Ability: database management systems administr...
2,Oracle Database Administrator,Ability: Over 4+ years of Experience as Archit...
3,Oracle Database Administrator,"Ability: Oracle Database Administration, Datab..."
4,Oracle Database Administrator,"Ability: Oracle Database Administration, Oracl..."


In [35]:
read_data(resume_dataset,'Category')


 __Aperçu des données :
                        Category  \
0         Database Administrator   
1         Database Administrator   
2  Oracle Database Administrator   

                                              Resume  
0  Ability: Installation and Building Server, Run...  
1  Ability: database management systems administr...  
2  Ability: Over 4+ years of Experience as Archit...  

 __Informations sur les données :
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12104 entries, 0 to 12103
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   Category  12104 non-null  object
 1   Resume    12104 non-null  object
dtypes: object(2)
memory usage: 189.2+ KB
None
 
 __Taille : 
 (12104, 2)

 __Information sur les Categories: 
                                 Category  count
0                       Security_Analyst    861
1                     Software_Developer    783
2       Web_Developer,Software_Developer    648
3          

### Normalisation et Preprocessing

#### Normalisation Categories

##### __categories separer par -

In [36]:

def normalize_categories(data,column_name):
    """Normalise les noms des catégories"""
    df = data.copy()
    
    # Normalisation de base
    df[column_name] = (
        df[column_name]
        .str.replace('_', ' ')
        .str.title()
        .str.strip()
    )
    
    # Gestion des variantes "Sr"
    df[column_name] = (
        df[column_name]
        .str.replace(r'\bSr\.?\b', 'Senior', regex=True)
        .str.replace(r'Senior\.', 'Senior', regex=True)
        .str.strip()
    )
    
    return df

In [37]:

# Avant traitement
print("Avant normalisation:")
print(resume_dataset['Category'].nunique(), "catégories uniques")
resume_dataset=normalize_categories(resume_dataset,'Category')
# Après traitement
print("\nAprès normalisation:")
print(resume_dataset['Category'].nunique(), "catégories uniques")


Avant normalisation:
220 catégories uniques

Après normalisation:
182 catégories uniques


##### __Gestion des catégories 


In [38]:
# Identification des catégories rares
category_counts = resume_dataset['Category'].value_counts()
rare_categories = category_counts[category_counts < 30].index
print(f"\nNombre de catégories rares (moins de 20 occurrences): {len(rare_categories)}")
print("\nListe des catégories rares avec leur compte :")
print(rare_categories.sort_values())


Nombre de catégories rares (moins de 20 occurrences): 136

Liste des catégories rares avec leur compte :
Index(['Administrative Assistant', 'Advocate', 'Aem Developer',
       'Android Application Developer', 'Android Developer',
       'Application Developer', 'Arts', 'Automation Testing',
       'Backend Developer', 'Blockchain',
       ...
       'SeniorFullstack Developer', 'Systems Analyst', 'Systems Engineer',
       'Technical Consultant', 'Technical Project Manager', 'Testing',
       'Ui Developer Ui Developer Ui Developer', 'Ui/ Front End Developer',
       'Web Designing', 'Web Developer Web Developer Web Developer'],
      dtype='object', name='Category', length=136)


In [39]:
## Analyse et Groupement categories de faible Occurrence
map_remplacement = {
    # --- Other IT  ---
    "Automation Testing": "Other IT",
    "Blockchain": "Other IT",
    "Director Of It": "Other IT",
    "Help Desk Analyst": "Other IT",
    "Help Desk Technician": "Other IT",
    "It Auditor": "Other IT",
    "Pmo": "Other IT",
    "Program Manager": "Other IT",
    "Technical Consultant": "Other IT",
    "Testing": "Other IT",
    "Desktop Support Technician": "Other IT",
    "It Support Specialist": "Other IT",
    "It Technician": "Other IT",
    "Project Coordinator": "Other IT",
    "It Director":"Other IT",
    
    "Scrum Master": "Project Manager",
    "Technical Project Manager": "Project Manager",

    # --- Other No IT (8) ---
    "Civil Engineer": "Other No IT",
    "Electrical Engineering": "Other No IT",
    "Mechanical Engineer": "Other No IT",
    "Administrative Assistant": "Other No IT",
    "Arts": "Other No IT",
    "Customer Service Representative": "Other No IT",
    "Health And Fitness": "Other No IT",
    "Sales": "Other No IT",
    "Security Officer": "Other No IT",
    "Hr": "Other No IT",
    "Sales Associate": "Other No IT",
    "Advocate": "Other No IT",
    # --- Big Data Cloud Engineer ---
    "Devops Engineer": "Big Data Cloud Developer",
    "Cloud Engineer":"Big Data Cloud Developer",
    "Hadoop": "Big Data Cloud Developer",
    "Hadoop Developer": "Big Data Cloud Developer",
    "Sr. Hadoop Developer": "Big Data Cloud Developer",
    "Etl Developer": "Big Data Cloud Developer",

    # --- Data Analyst ---
    "Data Analyst ": "Data Analyst ",
    "It Business Analyst": "Data Analyst",
    "Data Science":"Data Scientist",
     # --- Consultant---
    "Consultant Consultant":"Consultant",
    "Consultant Consultant Consultant":"Consultant",
    "Contractor Contractor Contractor":"Consultant",
    "Independent Contractor":"Consultant",

    "Mobile App Developer (Ios/Android)": "Mobile Developer",
    "Android Application Developer":"Mobile Developer",
    "Android Developer":"Mobile Developer",
    
    "Business Analyst": "Data Analyst",
    "It Analyst": "Data Analyst",
    "Salesforce Administrator'": "Salesforce Developer",
    "Salesforce Admin/ Developer": "Salesforce Developer",
    "Salesforce Lightning Developer": "Salesforce Developer",
    "Frontend Developer": "Front End Developer",
    # "Sr. Front End Developer": "Senior Front End Developer", 
    # "Sr. Ui Developer": "Senior Front End Developer",
    "Ui Developer Ui Developer Ui Developer": "UI Developer",  
    "Front- End Developer":"Front End Developer",
    "Front End Engineer":"Front End Developer",
    "Front End/Angular Developer":"Front End Developer",
    "Front End/Ui Developer":"Front End Developer",
    "Front- End Web Developer":"Front End Developer",
    "Front-End Web Developer":"Front End Developer",
    "Front-End Developer":"Front End Developer",
    "Developer Developer": "Software Developer",  
    "Lead Front End Developer":"Lead Developer",
    "Lead Java Developer": "Lead Developer",
    "Freelance Web Developer": "Freelance Developer",
    "Freelance Front End Developer":"Freelance Developer", 
    "Senior Front End Developer":"Front End Developer",
    "Senior Ui Developer":"UI Developer",
    "Ui Developer":"UI Developer",
    "Sql Database Administrator":"SQL/SQL Server Database Administrator",
    "Sql Server Database Administrator":"SQL/SQL Server Database Administrator",
    "Java Developer":"Java Full Stack Developer",
    "Full Stack Java Developer":"Java Full Stack Developer",
    "Senior Java Developer":"Senior Java Full Stack Developer",
    "Senior Java/J2Ee Developer":"Java/J2Ee Developer",
     
    #                    
    "It Security Engineer": "Security Engineer",                               
    
    "It Project Coordinator": "Project Manager",
    "Senior It Project Manager": "Senior Project Manager",
    "It Consultant": "Consultant",
    "It Specialist": "Other IT" ,
    "System Administrator":"Systems Administrator",
}
# Remplacement 
c_resume_dataset=resume_dataset.copy()
c_resume_dataset['Category'] = c_resume_dataset['Category'].replace(map_remplacement)
# Suppression des lignes où Category == "Web Developer,Software Developer"
c_resume_dataset = c_resume_dataset[c_resume_dataset['Category'] != "Web Developer,Software Developer"]

# Avant traitement
print("Avant groupement:")
print(resume_dataset['Category'].nunique(), "catégories uniques")
# Après traitement
print("\nAprès normalisation:")
print(c_resume_dataset['Category'].nunique(), "catégories uniques")
# # Vérification
# print("Nombre de catégories par groupe après remplacement :")
print(c_resume_dataset['Category'].value_counts())


Avant groupement:
182 catégories uniques

Après normalisation:
113 catégories uniques
Category
Network Administrator             1056
Security Analyst                   915
Software Developer                 853
Database Administrator             660
Project Manager                    649
                                  ... 
It Consultant/ Project Manager       7
Self Employed                        7
Data Security Analyst                7
Sap Developer                        6
Web Designing                        4
Name: count, Length: 113, dtype: int64


In [40]:
# Identifier les catégories à supprimer (<30 occurrences)
new_rare_categories = category_counts[(category_counts < 30) ].index
# Filtrer le DataFrame pour ne garder que les lignes avec des catégories valides
filtered_dataset = c_resume_dataset[~c_resume_dataset['Category'].isin(new_rare_categories)]

# Vérification
print(f"Taille originale : {len(c_resume_dataset)}")
print(f"\nNombre de catégories : {len(c_resume_dataset['Category'].value_counts())}")
print(f"Taille après filtrage : {len(filtered_dataset)}")
print(f"\nNombre de catégories : {len(filtered_dataset['Category'].value_counts())}")

Taille originale : 11456

Nombre de catégories : 113
Taille après filtrage : 10455

Nombre de catégories : 39


In [41]:
print("\nCatégories conservées :")
print(filtered_dataset['Category'].value_counts())


Catégories conservées :
Category
Network Administrator                    1056
Security Analyst                          915
Software Developer                        853
Database Administrator                    660
Project Manager                           649
Front End Developer                       578
Systems Administrator                     537
Java Full Stack Developer                 523
Python Developer,Software Developer       521
Python Developer                          423
Software Developer,Web Developer          413
Java Developer,Software Developer         388
Oracle Database Administrator             256
Project Manager,Software Developer        242
It Security Analyst                       230
Senior Java Full Stack Developer          223
It Project Manager                        204
Other IT                                  169
Senior Python Developer                   167
Job Seeker                                142
Network Engineer                          140


In [42]:
category_counts = filtered_dataset['Category'].value_counts()
rare_categories = category_counts[category_counts < 40].index
print(f"\nNombre de catégories avec moins de 40 enregistrement): {len(rare_categories)}")
print("\nListe des catégories rares avec leur compte :")
print(rare_categories.sort_values())


Nombre de catégories avec moins de 40 enregistrement): 3

Liste des catégories rares avec leur compte :
Index(['Freelance Developer', 'Senior Oracle Database Administrator',
       'Software Engineer'],
      dtype='object', name='Category')


#### Doublons

In [43]:
# Vérification des doublons
print(f"Nombre de doublons exacts: {filtered_dataset.duplicated().sum()}")

# Vérification des doublons potentiels (même texte mais catégories différentes)
duplicate_texts = filtered_dataset[filtered_dataset.duplicated(subset=['Resume'], keep=False)]
print(f"\nExemples de textes dupliqués avec catégories différentes:")
duplicate_texts.sort_values('Resume').head(3)

Nombre de doublons exacts: 0

Exemples de textes dupliqués avec catégories différentes:


,Category,Resume


In [44]:
# filtered_dataset.to_csv(data_dir/'processed/cleaned_combined_resume_1.csv')

#### Preprocessing de base dataset

In [45]:
def preprocess_text(text):
    # Mise en minuscules
    text = text.lower()
    # Suppression :
    text = re.sub(r'https?://\S+|www\.\S+', '', text) #des URLs
    text = re.sub(r'<[^>]+>', '', text)# des balises HTML
    #text = re.sub(r'[\[\]\|@#$%^&*~]', '', text) 
    text = re.sub(r'[\[\]\|@#$%^&*~_+=<>/\\{}¦©®™]', '', text)#caractères non-linguistiques
    text = re.sub(r'--+', ' ', text)  #les suites de tirets
    text = re.sub(r'\d{6,}', ' ', text) #des longues séquences numériques (6 chiffres et plus)
    text = emoji.replace_emoji(text, replace='') # Suppression des emojis
    text = re.sub(r'\s+', ' ', text).strip()     # des espaces multiples
    return text

In [46]:
def clean_dataset(data,column_name)->pd.DataFrame:
    new_data=data.copy(deep=True)
    new_data.loc[:, column_name]=new_data[column_name].apply(lambda x:preprocess_text(x))
    return new_data

In [47]:
cleaned_resume_dataset=clean_dataset(filtered_dataset,'Resume')
read_data(cleaned_resume_dataset,'Category')


 __Aperçu des données :
                        Category  \
0         Database Administrator   
1         Database Administrator   
2  Oracle Database Administrator   

                                              Resume  
0  ability: installation and building server, run...  
1  ability: database management systems administr...  
2  ability: over 4 years of experience as archite...  

 __Informations sur les données :
<class 'pandas.core.frame.DataFrame'>
Index: 10455 entries, 0 to 12103
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   Category  10455 non-null  object
 1   Resume    10455 non-null  object
dtypes: object(2)
memory usage: 245.0+ KB
None
 
 __Taille : 
 (10455, 2)

 __Information sur les Categories: 
                                  Category  count
0                   Network Administrator   1056
1                        Security Analyst    915
2                      Software Developer    853
3           

In [48]:
filtered_dataset.head(5)

,Category,Resume
0,Database Administrator,"Ability: Installation and Building Server, Run..."
1,Database Administrator,Ability: database management systems administr...
2,Oracle Database Administrator,Ability: Over 4+ years of Experience as Archit...
3,Oracle Database Administrator,"Ability: Oracle Database Administration, Datab..."
4,Oracle Database Administrator,"Ability: Oracle Database Administration, Oracl..."


In [49]:
#  catégories à supprimer
categories_to_remove = ['It Security Analyst', 'It Project Manager','Software Developer,Web Developer',
                        'Front End Web Developer','Senior Oracle Database Administrator','Freelance Developer']

# Suppression effective
cleaned_resume_dataset = cleaned_resume_dataset[~cleaned_resume_dataset['Category'].isin(categories_to_remove)].copy(deep=True)
print("Nombre  CV : ",len(cleaned_resume_dataset['Resume'].value_counts()))
print("Nombre  Categories : ",len(cleaned_resume_dataset['Category'].value_counts()))
print("Liste Categories : ",cleaned_resume_dataset['Category'].value_counts().index.sort_values())

Nombre  CV :  9497
Nombre  Categories :  33
Liste Categories :  Index(['Big Data Cloud Developer', 'Consultant', 'Cyber Security Analyst',
       'Data Scientist', 'Database Administrator', 'Front End Developer',
       'Full Stack Developer', 'Information Security Analyst', 'It Manager',
       'Java Developer,Software Developer', 'Java Full Stack Developer',
       'Java/J2Ee Developer', 'Job Seeker', 'Mobile Developer',
       'Network Administrator', 'Network Engineer',
       'Oracle Database Administrator', 'Other IT', 'Other No IT',
       'Project Manager', 'Project Manager,Software Developer',
       'Python Developer', 'Python Developer,Software Developer',
       'SQL/SQL Server Database Administrator', 'Security Analyst',
       'Senior Database Administrator', 'Senior Java Full Stack Developer',
       'Senior Python Developer', 'Software Developer', 'Software Engineer',
       'Systems Administrator', 'UI Developer', 'Web Developer'],
      dtype='object', name='Category'

#### Enregistrement Dataset

In [50]:
print("Nombre  CV : ",len(cleaned_resume_dataset['Resume'].value_counts()))
print("Nombre  Categories : ",len(cleaned_resume_dataset['Category'].value_counts()))

Nombre  CV :  9497
Nombre  Categories :  33


In [51]:

cleaned_resume_dataset.to_csv(data_dir/'processed/cleaned_combined_resume_final.csv')
# read_data(resume_dataset)

## 2. DATASET : OFFERS

In [52]:

offer_dataset=pd.read_csv(data_dir/'interim/combined_offer_dataset.csv')
offer_dataset.head()

,Job Title_Category,Job Description
0,Prompt Engineer and Librarian,"Anthropic’s mission is to create reliable, int..."
1,Social Sciences Librarian - 500421,Social Sciences Librarian - 500421\n\nSUMMARY:...
2,"Collection Management Specialist, EBSCO Books",EBSCO Information Services (EIS) provides a co...
3,Humanities Librarian,SUMMARY:\nThe Humanities Librarian serves as t...
4,Librarian,"Salary Range\n\n$49,705 - $95,595/annually\nWo..."


In [53]:
read_data(offer_dataset,'Job Title_Category')


 __Aperçu des données :
                              Job Title_Category  \
0                  Prompt Engineer and Librarian   
1             Social Sciences Librarian - 500421   
2  Collection Management Specialist, EBSCO Books   

                                     Job Description  
0  Anthropic’s mission is to create reliable, int...  
1  Social Sciences Librarian - 500421\n\nSUMMARY:...  
2  EBSCO Information Services (EIS) provides a co...  

 __Informations sur les données :
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3339 entries, 0 to 3338
Data columns (total 2 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   Job Title_Category  3339 non-null   object
 1   Job Description     3339 non-null   object
dtypes: object(2)
memory usage: 52.3+ KB
None
 
 __Taille : 
 (3339, 2)

 __Information sur les Categories: 
                                      Job Title_Category  count
0                                       

### Normalisation et Preprocessing

#### Normalisation

##### #Job Title_Category

In [54]:
# Avant traitement
print("Avant normalisation:")
print(offer_dataset['Job Title_Category'].nunique())
offer_dataset=normalize_categories(offer_dataset,'Job Title_Category')
# Après traitement
print("\nAprès normalisation:")
print(offer_dataset['Job Title_Category'].nunique())

Avant normalisation:
1773

Après normalisation:
1746


###### Gestion category:Mappage et reorganisation 

In [55]:
import nltk
from nltk.tokenize import word_tokenize
import string
import re

nltk.download('punkt_tab')


# Vérifier si tous les mots d'une phrase sont dans un titre (ordre non important)
def contains_all_words(title_tokens, phrase_words):
    return all(word in title_tokens for word in phrase_words)

# Convertir les titres en minuscules pour faciliter la comparaison
offer_dataset['Job Title_Category'] = offer_dataset['Job Title_Category'].str.lower()


#  catégorisation basée sur des mots-clés
def categorize_job_title(title):

    if isinstance(title, str):
        title_lower = title.lower()
        # Découpage  mots 
        title_tokens = word_tokenize(title_lower.translate(str.maketrans('', '', string.punctuation)))

        # correspondance avec les mots-clés
        if contains_all_words(title_tokens, ["data", "scientist"]):
            return "data scientist"
        elif contains_all_words(title_tokens, ["machine", "learning"]):
            return "machine learning engineer"
        elif contains_all_words(title_tokens, ["data", "analytics"]):
            return "data analyst"
        elif contains_all_words(title_tokens, ["data", "analyst"]):
            return "data analyst"
        elif contains_all_words(title_tokens, ["data", "engineer"]):
            return "data engineer"
        elif contains_all_words(title_tokens, ["software", "engineer"]):
            return "software engineer"
        elif contains_all_words(title_tokens, ["librarian"]):
            return "librarian"
        elif contains_all_words(title_tokens, ["actuarial", "analyst"]):
            return "actuarial analyst"
        elif contains_all_words(title_tokens, ["human", "resources"]):
            return "human resources"
        elif contains_all_words(title_tokens, ["financial", "analyst"]):
            return "financial analyst"
        elif contains_all_words(title_tokens, ["teacher"]):
            return "teacher"
        elif contains_all_words(title_tokens, ["data", "science"]):
            return "data scientist"
        elif contains_all_words(title_tokens, ["financial", "planning", "analys"]):
            return "financial planning analyst"
        elif contains_all_words(title_tokens, ["assistant", "professor"]):
            return "assistant professor"
        elif contains_all_words(title_tokens, ["ml"]):
            return "ml engineer"
        elif contains_all_words(title_tokens, ["business", "analyst"]):
            return "business analyst"
        elif contains_all_words(title_tokens, ["actuarial", "associate"]):
            return "actuarial associate"
        elif contains_all_words(title_tokens, ["electrical ", "engineer"]):
            return "electrical engineer"
        elif contains_all_words(title_tokens, ["project", "manager"]):
          return "project manager"
        elif contains_all_words(title_tokens, ["backend", "engineer"]):
          return "backend engineer"
        else:
            return title  # Garder l'original si aucun mot-clé ne correspond
    return title  # Retourner l'original si ce n'est pas du texte


# Appliquer la catégorisation
offer_dataset['Job Title_Category'] = offer_dataset['Job Title_Category'].apply(categorize_job_title)
print(f"Nombre de Categories': {offer_dataset['Job Title_Category'].nunique()}")
# Afficher un aperçu des données avec la nouvelle catégorisation
display(offer_dataset.head(10))

[nltk_data] Error loading punkt_tab: Remote end closed connection
[nltk_data]     without response


Nombre de Categories': 684


,Job Title_Category,Job Description
0,librarian,"Anthropic’s mission is to create reliable, int..."
1,librarian,Social Sciences Librarian - 500421\n\nSUMMARY:...
2,"collection management specialist, ebsco books",EBSCO Information Services (EIS) provides a co...
3,librarian,SUMMARY:\nThe Humanities Librarian serves as t...
4,librarian,"Salary Range\n\n$49,705 - $95,595/annually\nWo..."
5,librarian,Introduction\nPrince William Public Libraries ...
6,librarian,Duties\nCatalogs a variety of materials in an ...
7,librarian,"Salary\n$48,401.00 - $72,602.00 Annually\nLoca..."
8,librarian,The Position\n\nLibrarian\nThe City of Ontario...
9,librarian,Librarian C—Special Collections Processing Lib...


In [56]:
# Mapping  des catégories 
category_mapping = {
    # IT Rôles
    "Data Scientist": [
        "data scientist", "applied scientist", "research scientist",
        "interdisciplinary-microbiologist/data scientist", "data scientist-health",
        "principal-data scientist", "data scientists","data scientist python"
    ],
    "AI/ML Specialist": [
        "ai/ml", "machine learning", "ml engineer", "ai/ml specialist","ml/ai engineer with nlp specialization",
        "ai engineer", "ml/ai engineer", "ai researcher", "nlp", "computer vision",
        "gen ai/ml", "ai/ml developer", "ai/ml researcher", "ai training for writers",
        "ai/machine learning developer", "ai research scientist", "ai research scientist: aec",
        "applied ai scientist / engineer", "computer vision / deep learning scientist", "ai automation", "ai agent"
    ],
    "Data Analyst": [
        "data analyst", "analytics analyst", "insights analyst", "powerbi",
        "analyst, analytics", "analytics associate", "customer insights analyst",
        "officer - real time analytics", "analyst-data science", "analyst, analytics",
        "powerbi analyst", "insight analyst uk", "operation analyst-power bi & sql",
        "analyst-data analytics", "quant analytics analyst", "decision scientist",
        "report/data analyst", "analyst", "data analyst/engineer", "report writer-data analyst",
        "internship - data analysis","associate - data researcher", "analytics scientist",
        "business analyst", "business intelligence",
        "business intelligence analyst/developer", "business intelligence analyst",
        "business intelligence intern", "senior bi reporting developer"
    ],
    "Data Engineer": [
        "data engineer", "etl", "data pipeline", "database specialist",
        "data modeler- life sciences", "data architect", "data modeler",
        "analytics engineer", "data engineer(aws)", "sector analyst, mi-data",
        "data associate - gurgaon", "data analyst/engineer", "data modeler (analytical systems)",
        "data/software engineer", "data engineer(h/f)","data intern"
        "data architect","data modeler","data strategist","database administrator"
    ],
    "Software Developer": [
        "software engineer", "software developer", "full stack",
        "frontend developer", "qa/test engineer", "software development",
        "junior software development", "associate software development",
        "test engineer", "front end developer i", "react developer",
        "software/ai engineer", "full stack developer",
        "remote elixir/phoenix developer", "frontend engineer",
        "backend engineer", "nodejs back end developer","senior angular developer",
        "apache solr developer","senior web developer (angular & .net) - remote",
        "senior fullstack engineer","backend developer, mid to senior level (golang) fintech",
        "senior specialist, back-end, mainframe developer","c# developer", "principal programmer", "rpa developer",
        "bi developer", "automation testing"
    ],
    "Python / Java Developer": [
        "python developer", "python sql developer","django developer", "flask developer", "python backend",
        "analytics with python","senior - python+sql","senior automation testing engineer (python)", 
        "jr. java developer", "java/j2ee developer","java developer", "java/nodejs developer",
        "spring developer", "java backend", "javafx developer",
        "android developer java", "java microservices", "java engineer"
    ],
    "DevOps/Cloud": [
        "devops", "cloud", "aws", "azure", "linux administrator",
        "senior azure & epic ai engineer", "usefulbi corporation - senior devops engineer"
    ],
    "Hardware Engineer": [
        "hardware engineer", "electrical engineer", "electronics",
        "electrician/technician", "reliability engineer", "product design engineer",
        "hardware design engineer", "electronics design engineer"
    ],
    "Mechanical Engineer": [
        "mechanical engineer", "mechanical designer", "mechanical design engineer",
        "mechanical engineers/designers", "staff engineer structural",
        "advanced mechanical design engineer", "package design and modeling engineer",
        "senior mechanical / materials engineer","staff engineer - mechanical",
        "mechanical project engineer","r&d engineer", "graduate façades engineer",
        "mechanical and process engineer"
    ],

    # Non-IT Rôles
    "Librarian/Archivist": ["librarian", "archivist", "collection management specialist"],
    "Teacher/Professor": [
        "teacher", "professor", "instructor", "educator",
        "early childhood educator/teacher", "assistant/associate adjunct faculty",
        "core faculty- bachelor of science early education"
    ],
    "Counselor": [
        "counselor", "guidance counselor", "school psychologist",
        "post secondary counselor", "professional school counselor"
    ],
    "Financial Specialist" : [
    "accounting associate", "actuarial qa", "financial planning",
    "investment banker", "valuation consultant", "investment strategist",
    "private banker", "commercial banker", "automotive finance",
    "controller", "banker i", "high net worth","financial analyst", "fp&a", "investment analyst",
    "investment banking associate", "jr investment analyst / trader",
    "financial analyst/staff accountant", "senior operations finance analyst",
    "commercial real estate analyst", "plan, forecast, budget analyst",
    "finance analyst ii", "budget analyst", "analyst-risk & info management"
    ],
    "Actuarial Specialist": [
        "actuarial analyst", "actuary", "actuarial associate",
        "assistant actuarial analyst", "actuarial development program - level i",
        "actuarial assistant", "associate actuary", "model validation actuary"
    ],
    "HR Professional": [
        "human resources", "hr manager", "talent acquisition", "hr",
        "hr generalist", "human resource generalist", "regional human resource manager",
        "hr assistant", "bilingual hr coordinator", "equal employment specialist",
        "human resource business partner i", "human capital management specialist"
    ],
    "Legal Counsel": [
        "counsel", "attorney", "paralegal", "legal assistant",
        "law clerk", "judicial clerk 2", "corporate litigation counsel",
        "assistant general counsel - retail", "government information specialist",
        "assistant city prosecutor i", "assistant general counsel - us consumer retail bank",
        "compliance counsel", "freelance lawyer", "entry level family law associate",
        "hiring legal gladiators", "privacy counsel", "corporate counsel, transactions",
        "commercial counsel", "associate counsel"
    ],
    "Chef/Culinary": [
        "chef", "sous chef", "kitchen manager", "culinary",
        "executive chef", "residential chef", "corporate chef", "private chef",
        "banquet chef", "development chef", "cdc/production chef", "chef & b"
    ],
    "Bartender/Server": [
        "bartender", "server", "mixologist", "fine dining bartender",
        "service bartender", "servers", "server/bartender", "beverage cart attendant",
        "restaurant server/bartender"
    ],
    "Loan Officer": [
        "loan officer", "mortgage officer", "commercial loan",
        "jr commercial loan officer", "licensed loan officer assistant",
        "mortgage loan officer assistant", "junior loan officer partner"
    ],
    "Executive/C-Suite": [
        "ceo", "cto", "cfo", "chief executive", "vp", "director",
        "managing director/director of diversity", "director, network and community building",
        "senior product manager, auto finance", "director of engineering | travel",
        "director, marketing", "director of sales - peninsula del rey"
    ],
    "Research Scientist": [
        "hydrogen/tritium materials scientist (experienced)",
        "research scientist", "r&d scientist", "purification scientist"
        "program specialist",
        "senior research associate/ scientist, ngs prep & molecular genomics",
        "research and development scientist",
        "scientist - molecular biology",
        "ngs scientist",
        "scientist - biomarker and flow cytometry",
        "analytics manager",
        "staff scientist- upstream pd",
        "senior scientist - extractables & leachables",
        "computational scientist",
        "production engineer - statistics/data analysis",
        "statistical scientist",
        "computational behavioral scientist",
        "real world evidence (rwe) scientist",
        "development scientist, voltaren",
        "weapons and sensors engineer/scientist",
        "applied computer scientist",
        "senior scientist - toxicologist - product integrity (stewardship)",
        "scientist / group lead, cancer biology","scientist - translational", "purification scientist",
        "computer scientist", "clinical laboratory", "field application",
        "metabolic engineering", "hydrogen/tritium", "ngs prep",
        "bioinformatics", "medical lab"
        "scientist/research associate-metabolic engineering"],
    "Other It Professional": ["project manager", "technical program manager", "renewable energy",
        "district operations manager", "construction manager assistant",
        "product operations manager", "manager, e-commerce" ,"ui/ux","systems engineer", "network engineer", "it system",
        "system analyst", "cyber security analyst", "senior systems engineer",
        "lead network engineer",
        "projects engineer integrated systems",
        "senior systems engineer"              
    ],
    "Other No IT Professionnel":[ 
        "sales associate", "account manager", "relationship banker",
        "inside sales representative", "regional sales manager",
        "account manager - landscaping", "oakley - specialized consultant"
        ,"electrician", "electrical technician", "avionics",
        "electrical controls engineer", "electronics technician",
        "senior electrical design engineer", "electrical engineer - solar",
        "facilities electrical engineer", "remote service engineer neta","sales associate", "account manager", "relationship banker",
        "inside sales representative", "regional sales manager",
        "account manager - landscaping", "oakley - specialized consultant","renewable energy", "solar engineer", "wind energy",
        "project engineer-renewable energy", "renewable energy systems engineer",
        "staff engineer, renewable procurement (solar)", "offshore wind project engineers"]
}

# Fonction  de mappage
def map_category(title):
    if not isinstance(title, str):
        return title
    title_lower = title.lower()

    # Recherche prioritaire des catégories spécifiques
    for category, keywords in category_mapping.items():
        for keyword in keywords:
            if keyword in title_lower:
                return category
    return title

offer_dataset['Job Title_Category'] = offer_dataset['Job Title_Category'].apply(map_category)

# Afficher les résultats
print(f"Nombre de catégories réduit à: {offer_dataset['Job Title_Category'].nunique()}")
print(offer_dataset['Job Title_Category'].value_counts())

Nombre de catégories réduit à: 168
Job Title_Category
AI/ML Specialist                                           714
Data Scientist                                             704
Data Analyst                                               658
Software Developer                                         306
Data Engineer                                              275
                                                          ... 
manufacturing engineer- entry level                          1
manufacturing process engineer                               1
mechanical run plant engineer                                1
ba                                                           1
vice president, biometrics and clinical data management      1
Name: count, Length: 168, dtype: int64


###### Suppression category faible occurence

In [57]:
categories_to_remove = ['analista de dados', '研究所-数据岗', 'ejecutivo', 'cientista', 'high voltage','analista']

offer_dataset = offer_dataset[~offer_dataset['Job Title_Category'].str.lower().str.contains('|'.join(categories_to_remove), na=False)]
job_category_counts = offer_dataset['Job Title_Category'].value_counts()

low_frequency_categories = job_category_counts[job_category_counts <3]
print("Job Title Categories avec moins de  3 occurrences:")
display(len(low_frequency_categories.index))
display(low_frequency_categories)
offer_dataset=offer_dataset[~offer_dataset['Job Title_Category'].isin(low_frequency_categories.index)]
print(f"Nombre de catégories après suppression': {offer_dataset['Job Title_Category'].nunique()}")
print(f"catégories après suppression:{offer_dataset['Job Title_Category'].value_counts()}")

Job Title Categories avec moins de  3 occurrences:


116

Job Title_Category
multiple positions                                         2
energy engineer                                            2
babysitting and childcare                                  2
programmer                                                 2
producer                                                   2
                                                          ..
electronic graphics operator, cnn                          1
planning engineer - new construction                       1
cad designer                                               1
high- voltage dielectric materials scientist               1
vice president, biometrics and clinical data management    1
Name: count, Length: 116, dtype: int64

Nombre de catégories après suppression': 23
catégories après suppression:Job Title_Category
AI/ML Specialist             714
Data Scientist               704
Data Analyst                 658
Software Developer           306
Data Engineer                275
Chef/Culinary                 67
Teacher/Professor             57
Legal Counsel                 54
Librarian/Archivist           47
HR Professional               44
Mechanical Engineer           43
Research Scientist            29
Hardware Engineer             26
Bartender/Server              23
DevOps/Cloud                  22
Other It Professional         19
Executive/C-Suite             17
Counselor                     16
Financial Specialist          15
Other No IT Professionnel     15
Python / Java Developer       14
Actuarial Specialist          12
Loan Officer                  11
Name: count, dtype: int64


##### #Job Description

In [58]:
# Vérification des doublons
print(f"Nombre de doublons exacts: {offer_dataset.duplicated().sum()}")

# Vérification des doublons potentiels (même texte mais catégories différentes)
duplicate_texts = offer_dataset[offer_dataset.duplicated(subset=['Job Description'], keep=False)]
print(f"\nExemples de textes dupliqués avec catégories différentes:")
duplicate_texts.sort_values('Job Description').head(3)

Nombre de doublons exacts: 11

Exemples de textes dupliqués avec catégories différentes:


,Job Title_Category,Job Description
1619,Data Engineer,"At Lyft, our purpose is to serve and connect. ..."
1321,Data Engineer,"At Lyft, our purpose is to serve and connect. ..."
1063,Software Developer,Description\nAbout us\nInnovation is fuelled b...


##### Duplication

In [59]:
offer_dataset = offer_dataset.drop_duplicates()

# Supprimer les doublons basés sur la description de poste
offer_dataset = offer_dataset.drop_duplicates(subset=['Job Description'], keep='first')

# Vérification finale
print(f"\nNombre final de lignes: {len(offer_dataset)}")
print(f"Nombre de doublons exacts restants: {offer_dataset.duplicated().sum()}")
print(f"Nombre de descriptions dupliquées restantes: {offer_dataset.duplicated(subset=['Job Description']).sum()}")


Nombre final de lignes: 3172
Nombre de doublons exacts restants: 0
Nombre de descriptions dupliquées restantes: 0


#### Preprocessing

In [60]:
offer_dataset.head(5)

,Job Title_Category,Job Description
0,Librarian/Archivist,"Anthropic’s mission is to create reliable, int..."
1,Librarian/Archivist,Social Sciences Librarian - 500421\n\nSUMMARY:...
2,Librarian/Archivist,EBSCO Information Services (EIS) provides a co...
3,Librarian/Archivist,SUMMARY:\nThe Humanities Librarian serves as t...
4,Librarian/Archivist,"Salary Range\n\n$49,705 - $95,595/annually\nWo..."


In [61]:
cleaned_offer_dataset=clean_dataset(offer_dataset,'Job Description')
read_data(cleaned_offer_dataset,'Job Title_Category')


 __Aperçu des données :
    Job Title_Category                                    Job Description
0  Librarian/Archivist  anthropic’s mission is to create reliable, int...
1  Librarian/Archivist  social sciences librarian - summary: working w...
2  Librarian/Archivist  ebsco information services (eis) provides a co...

 __Informations sur les données :
<class 'pandas.core.frame.DataFrame'>
Index: 3172 entries, 0 to 3338
Data columns (total 2 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   Job Title_Category  3172 non-null   object
 1   Job Description     3172 non-null   object
dtypes: object(2)
memory usage: 74.3+ KB
None
 
 __Taille : 
 (3172, 2)

 __Information sur les Categories: 
            Job Title_Category  count
0            AI/ML Specialist    710
1              Data Scientist    704
2                Data Analyst    658
3          Software Developer    300
4               Data Engineer    271
5               Che

#### Enregistrement Dataset

In [62]:
cleaned_offer_dataset.to_csv(data_dir/'processed/cleaned_combined_offer_final.csv')